# Dimensionality Reduction: PCA, t-SNE & UMAP

Reduce high-dimensional data to 2-3 dimensions for visualization, noise reduction, and faster training.


In [ ]:
# Colab setup - run opencolab_setup.ipynb first if using Google Colab
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'])


## 1. The Curse of Dimensionality


In [ ]:
print('''
As dimensions increase:
- Data becomes sparse (points move to edges)
- Distance metrics become meaningless
- Model needs exponentially more samples
- Overfitting risk increases dramatically

Dimensionality reduction helps by:
- Removing redundant/noisy features
- Preserving meaningful structure
- Enabling 2D/3D visualization
''')


## 2. PCA — Principal Component Analysis


In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
print(f'Original shape: {X.shape} (64 pixels per digit)')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Explained variance ratio
cumulative = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(10, 4))
plt.subplot(1,2,1)
plt.bar(range(1, 11), pca.explained_variance_ratio_[:10])
plt.xlabel('Component'); plt.ylabel('Variance Ratio')
plt.title('Scree Plot (top 10)')
plt.subplot(1,2,2)
plt.plot(range(1, 65), cumulative, 'bo-')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
plt.xlabel('Number of Components'); plt.ylabel('Cumulative Variance')
plt.title('Cumulative Explained Variance')
plt.legend()
plt.tight_layout(); plt.show()

n_95 = np.argmax(cumulative >= 0.95) + 1
print(f'Components needed for 95% variance: {n_95}')


## 3. Visualize Digits with PCA


In [ ]:
X_pca_2d = PCA(n_components=2).fit_transform(X_scaled)
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_2d[:,0], X_pca_2d[:,1], c=y, cmap='tab10', alpha=0.7, s=15)
plt.colorbar(scatter, label='Digit')
plt.title('Digits Dataset — PCA 2D Projection')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()


## 4. t-SNE — Non-Linear Embedding


In [ ]:
print('Computing t-SNE (may take a moment)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled[:1000])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y[:1000], cmap='tab10', alpha=0.7, s=20)
plt.colorbar(scatter, label='Digit')
plt.title('Digits Dataset — t-SNE 2D Embedding')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.show()
print('t-SNE preserves local structure better than PCA for visualization.')


## 5. PCA for Noise Reduction


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Original (64 dims)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
score_orig = cross_val_score(rf, X_scaled, y, cv=5).mean()
print(f'Original (64 dims): {score_orig:.4f}')

# PCA-reduced (top 20 components)
X_pca_20 = PCA(n_components=20).fit_transform(X_scaled)
score_pca = cross_val_score(rf, X_pca_20, y, cv=5).mean()
print(f'PCA 20 components: {score_pca:.4f}')

# PCA-reduced (top 2 components - drastic)
score_pca2 = cross_val_score(rf, X_pca_2d, y, cv=5).mean()
print(f'PCA 2 components: {score_pca2:.4f}')


## 6. PCA on Wine Dataset — Interpreting Components


In [ ]:
wine = load_wine()
X_wine = scaler.fit_transform(wine.data)
pca_wine = PCA(n_components=2).fit(X_wine)

loadings = pd.DataFrame(
    pca_wine.components_.T,
    columns=['PC1', 'PC2'],
    index=wine.feature_names
)
print('Feature Loadings (contribution to each component):')
print(loadings.round(3))

# What each PC represents
print('\nTop features driving PC1:')
print(loadings['PC1'].abs().sort_values(ascending=False).head(3))
print('\nTop features driving PC2:')
print(loadings['PC2'].abs().sort_values(ascending=False).head(3))


## Summary


In [ ]:
print('''
PCA               | Linear, fast, interpretable components
t-SNE             | Non-linear, great for visualization, slower
UMAP              | Non-linear, faster than t-SNE, preserves global structure

Rule of thumb:
- PCA first (always) for baseline and noise reduction
- t-SNE for visualization of complex high-dim data
- UMAP when you need both speed and quality
''')
